# FX matrix (quotes, triangulation, cross rates)

Deep-dive reference: **`FxMatrix`** for **direct** FX quotes and **policy-aware** lookups.

## Concept

Quotes are stored as **1 base = rate quote** (e.g. **EUR/USD = 1.10** means one euro buys 1.10 dollars). **Direct** pairs are those you explicitly **`set_quote`**; the **reciprocal** of a direct quote is served automatically.

**Cross rates** are resolved by the matrix itself: when no direct or reciprocal quote exists, `FxMatrix` **triangulates** through its configured **pivot currency** (USD by default), i.e. `EUR/GBP = (EUR/USD) x (USD/GBP)`. Never divide two quotes by hand in application code — the matrix owns the base/quote orientation, and `FxRateResult.triangulated` tells you when the pivot route was used.

## API walkthrough

`set_quote(base, quote, rate)` then `rate(base, quote, as_of_date, FxConversionPolicy.…)` returns **`FxRateResult`** with **`.rate`** and **`.triangulated`**.

In [1]:
from datetime import date

from finstack_quant.core.currency import Currency
from finstack_quant.core.market_data import FxConversionPolicy, FxMatrix

fx = FxMatrix()
fx.set_quote(Currency("EUR"), Currency("USD"), 1.10)
fx.set_quote(Currency("GBP"), Currency("USD"), 1.27)
fx.set_quote(Currency("JPY"), Currency("USD"), 0.0067)

as_of = date(2024, 1, 1)
policy = FxConversionPolicy.CASHFLOW_DATE

eur_usd = fx.rate(Currency("EUR"), Currency("USD"), as_of, policy)
print("Direct EUR/USD rate:", eur_usd.rate, "triangulated:", eur_usd.triangulated)

jpy_usd = fx.rate(Currency("JPY"), Currency("USD"), as_of, policy)
print("Direct JPY/USD (USD per 1 JPY):", jpy_usd.rate)

usd_jpy = fx.rate(Currency("USD"), Currency("JPY"), as_of, policy)
print("Inverse USD/JPY (JPY per 1 USD):", usd_jpy.rate)

Direct EUR/USD rate: 1.1 triangulated: False
Direct JPY/USD (USD per 1 JPY): 0.0067
Inverse USD/JPY (JPY per 1 USD): 149.2537313432836


## Practical example

**Multi-currency setup** with everything quoted against **USD**, then **EUR/GBP** and **CHF/EUR** read straight off the matrix. Neither cross is quoted directly, so the matrix routes each through the **USD pivot** and flags the result with **`.triangulated = True`**.

Inverses (e.g. USD/JPY derived from an explicit JPY→USD quote) are also automatic. Triangulation uses a **single** configured pivot — a pair whose pivot legs are not both available still raises `KeyError`, and a pair whose market convention routes through some other vehicle currency needs an explicit `set_quote`.

In [2]:
from datetime import date

from finstack_quant.core.currency import Currency
from finstack_quant.core.market_data import FxConversionPolicy, FxMatrix

fx = FxMatrix()
fx.set_quote(Currency("EUR"), Currency("USD"), 1.10)
fx.set_quote(Currency("GBP"), Currency("USD"), 1.27)
fx.set_quote(Currency("CHF"), Currency("USD"), 1.12)

as_of = date(2024, 1, 1)
pol = FxConversionPolicy.CASHFLOW_DATE

# Every pair below is asked of the matrix directly: it decides whether the answer
# is a stored quote, a reciprocal, or a cross triangulated through the USD pivot.
for base, quote in [("EUR", "USD"), ("EUR", "GBP"), ("CHF", "EUR"), ("GBP", "CHF")]:
    res = fx.rate(Currency(base), Currency(quote), as_of, pol)
    route = "triangulated via USD pivot" if res.triangulated else "direct or reciprocal quote"
    print(f"{base}/{quote}: {res.rate:.6f}  ({route})")

# A cross whose second pivot leg is missing has no route: the lookup raises
# (missing market data maps to KeyError) rather than returning a wrong number.
try:
    fx.rate(Currency("EUR"), Currency("SEK"), as_of, pol)
except KeyError as err:
    print("\nEUR/SEK (no SEK quote):", type(err).__name__, "—", err.args[0])

EUR/USD: 1.100000  (direct or reciprocal quote)
EUR/GBP: 0.866142  (triangulated via USD pivot)
CHF/EUR: 1.018182  (triangulated via USD pivot)
GBP/CHF: 1.133929  (triangulated via USD pivot)

EUR/SEK (no SEK quote): KeyError — FX triangulation failed for EUR->SEK via USD: USD->SEK rate not found


## Takeaways

- **`set_quote`** defines **base → quote** with **1 base = rate quote**.
- **`FxRateResult`** carries **`.rate`** and **`.triangulated`** — always read the flag rather than assuming a route.
- **Crosses** are produced by the matrix through its **pivot currency** (USD by default). Ask `fx.rate(...)` for the cross; do **not** divide two quotes yourself — that is where base/quote inversions creep in.
- Triangulation is **single-hop** through one pivot. If either pivot leg is unavailable the lookup raises `KeyError`; seed a **direct** quote for pairs that convention routes through a different vehicle.
- **`FxConversionPolicy`** selects **how** the as-of is interpreted for more complex instruments.